# Generate synthetic photometry with GaiaXPy

GaiaXPy provides the function `generate` to compute synthetic photometry from Gaia XP mean spectra. This tutorial shows how to select input spectra, choose one or more photometric systems, generate photometry, and inspect the results.

**Contents**

- [Input spectra](#gaiaxpy-input-spectra)
- [Choose one or more photometric systems](#gaiaxpy-choose-one-or-more-photometric-systems)
- [Generate and inspect photometry](#gaiaxpy-generate-and-inspect-photometry)
- [Standardised photometry](#gaiaxpy-standardised-photometry)
- [Trace the configuration and find the paper reference](#gaiaxpy-trace-configuration)
- [Choose another input type](#gaiaxpy-choose-another-input-type)
- [Save the output](#gaiaxpy-save-the-output)
- [Use additional photometric systems](#gaiaxpy-use-an-additional-photometric-system)
- [Other options you may need](#gaiaxpy-other-options-you-may-need)
- [Further reading](#gaiaxpy-further-reading)

<a id="gaiaxpy-input-spectra" name="gaiaxpy-input-spectra"></a>

## Input spectra

This tool requires Gaia XP mean spectra in their continuous representation. You can supply them as:

- A local CSV, ECSV, FITS, XML/VOTable file, including supported gzip-compressed inputs and Archive ZIP downloads.
- A pandas DataFrame, or a Series containing one spectrum row.
- A list or tuple of Gaia Archive source IDs (as integers or integer strings).
- An ADQL query that selects source IDs.

The first example uses a local file. Replace the example path with your file's path. For other inputs, see [Choose another input type](#gaiaxpy-choose-another-input-type).

**In order to run this tutorial, replace `spectra_input` with an existing file.**

In [ ]:
spectra_input = "path/to/dr3_xp_mean_spectra.ecsv"

<a id="gaiaxpy-choose-one-or-more-photometric-systems" name="gaiaxpy-choose-one-or-more-photometric-systems"></a>

## Choose one or more photometric systems

`PhotometricSystem.get_available_systems` lists the available systems by Gaia release, with additional systems in a separate group when loaded.

In [ ]:
from gaiaxpy import PhotometricSystem, generate

PhotometricSystem.get_available_systems()

System names ending in `_Std` identify standardised systems (see [Standardised photometry](#gaiaxpy-standardised-photometry)). Pass one system or a list of systems.

```python
ps = PhotometricSystem.DR3_PanSTARRS1
systems = [PhotometricSystem.DR3_JKC, PhotometricSystem.DR3_SDSS]
```

The example below selects Johnson–Kron–Cousins.

In [ ]:
systems = PhotometricSystem.DR3_JKC  # Using PhotometricSystem["DR3_JKC"] is equivalent

<a id="gaiaxpy-generate-and-inspect-photometry" name="gaiaxpy-generate-and-inspect-photometry"></a>

## Generate and inspect photometry

`generate` returns a pandas DataFrame with one row per source. Use `save_file=False` to avoid saving the output to a file.

In [ ]:
photometry = generate(spectra_input, photometric_system=systems, save_file=False)

The column names tell you the system, quantity and passband. For example, a column ending in `_mag_V` gives the synthetic V-band magnitude. The corresponding `_flux_V` and `_flux_error_V` columns give its flux and flux error. Use `source_id` to identify each source.

In [ ]:
print(photometry.columns.tolist())
photometry.head()

To try a different system, choose another name for your data release from the list above, then rerun the generation and inspection cells.

<a id="gaiaxpy-standardised-photometry" name="gaiaxpy-standardised-photometry"></a>

## Standardised photometry

Several DR3 photometric systems have `_Std` counterparts. The standardised fluxes use configured per-band offsets (see [Gaia Collaboration, Montegriffo et al. (2023)](https://doi.org/10.1051/0004-6361/202243709)).

The example compares ordinary DR3 JKC photometry with its standardised counterpart.

In [ ]:
standardised_system = PhotometricSystem.DR3_JKC_Std

standardised = generate(spectra_input,
                        photometric_system=[PhotometricSystem.DR3_JKC, standardised_system],
                        save_file=False)

raw_flux = "DR3_Jkc_flux_V"
std_flux = "DR3_JkcStd_flux_V"
comparison = standardised[["source_id", raw_flux, std_flux]].copy()
comparison["delta_flux_V"] = comparison[std_flux] - comparison[raw_flux]
comparison.head()

**Inspect the configured offsets.** Use the bands and offsets together to see which correction applies to each passband. The run metadata identifies the configuration used.

In [ ]:
dict(zip(standardised_system.get_bands(), standardised_system.get_offsets()))


<a id="gaiaxpy-trace-configuration" name="gaiaxpy-trace-configuration"></a>

## Trace the configuration and find the paper reference

For traceability, the metadata embedded in a photometric-system configuration can be accessed directly from the corresponding `PhotometricSystem` member. Where supplied by the configuration, this includes its version, description, change history, internal identifiers and paper reference.

In [ ]:
photometric_system_metadata = standardised_system.get_metadata()
photometric_system_metadata

The output metadata identifies the configuration file used for standardisation. Configuration metadata may contain a `reference` field identifying the corresponding scientific publication. The cell below shows the available references.

In [ ]:
metadata = standardised.attrs["gaiaxpy_metadata"]
standardisation = metadata["standardisation"]
standardisation_files = {entry["file"] for entry in standardisation}

for config in metadata["configuration_files"]:
    if config["file"] in standardisation_files:
        reference = config["metadata"].get("reference")
        if reference:
            print(config["file"])
            print(reference)

standardisation

<a id="gaiaxpy-choose-another-input-type" name="gaiaxpy-choose-another-input-type"></a>

## Choose another input type

You can pass CSV, ECSV, FITS or XML/VOTable files, including supported gzip-compressed files and Gaia Archive ZIP downloads. You can also pass a DataFrame or a pandas Series.

```python
photometry = generate(spectra, photometric_system=systems, save_file=False)
one_source = generate(spectra.iloc[0], photometric_system=systems, save_file=False)
```

For Gaia Archive input, pass source IDs as a list or tuple, or an ADQL query selecting source IDs.

```python
source_ids = [5853498713190525696, 5762406957886626816]
query = """
SELECT TOP 2 source_id
FROM gaiadr3.gaia_source
WHERE has_xp_continuous = 'True'
ORDER BY source_id
"""
by_id = generate(source_ids, photometric_system=systems, save_file=False)
by_query = generate(query, photometric_system=systems, save_file=False)
```


<a id="gaiaxpy-save-the-output" name="gaiaxpy-save-the-output"></a>

## Save the output

Set `save_file=True` and supply an output directory, a filename stem and a format. GaiaXPy still returns the DataFrame. Run this example when you want to write the CSV and its metadata.

```python
saved_photometry = generate(
    spectra_input,
    photometric_system=systems,
    save_file=True,
    output_path="results",
    output_file="synthetic_photometry",
    output_format="csv"
)
```

This writes `results/synthetic_photometry.csv` and `results/synthetic_photometry_metadata.csv`. For every other output format, the metadata is in the output file itself. Supported output formats also include ECSV, FITS and XML/VOTable.

<a id="gaiaxpy-use-an-additional-photometric-system" name="gaiaxpy-use-an-additional-photometric-system"></a>

## Use additional photometric systems

GaiaXPy supports user-defined DR3 filters from compatible XML filter files. The [SVO Filter Profile Service](https://svo2.cab.inta-csic.es/theory/fps/index.php) provides GaiaXPy base XML files for some filters.

Pass a single filter file or a directory to `load_additional_systems`. A directory is searched recursively. DR3 filter filenames use `<system>.gaiaxpy_dr3_<version>.xml`, and registered names include their versions (for example, `USER_DR3_Example_v1`). Replace the example path with an actual compatible file.

```python
from gaiaxpy import load_additional_systems

load_additional_systems("Example.gaiaxpy_dr3_v1.xml")
# Alternatively: load_additional_systems("path/to/filter_directory")
# Legacy behaviour PhotometricSystem = load_additional_systems("path/to/filter_directory") also works
print(PhotometricSystem.get_available_systems()["Additional systems"])

extra_system = PhotometricSystem.USER_DR3_Example_v1
extra_photometry = generate(spectra_input, photometric_system=extra_system, save_file=False)
```

You may select built-in and additional DR3 systems together.

<a id="gaiaxpy-other-options-you-may-need" name="gaiaxpy-other-options-you-may-need"></a>

## Other options you may need

- **Retain input columns.** Use `additional_columns` to carry existing input fields into the output, for example `additional_columns=["bp_chi_squared"]` when present.
- **Standardised photometry.** Select the corresponding DR3 `_Std` system where available; its configured offsets affect the generated photometry.
- **Flux-error correction and truncation.** `error_correction` applies the supported flux error corrections, and `truncation` uses the recommended number of basis functions. Processing metadata records relevant options and configurations.

<a id="gaiaxpy-further-reading" name="gaiaxpy-further-reading"></a>

## Further reading

- [Synthetic photometry error correction](https://gaia-dpci.github.io/GaiaXPy-website/tutorials/Error%20correction%20tutorial.html) explains when and how to apply flux-error corrections.

**Tracing a result.** If you need to reproduce a run or explain why two runs differ, check `photometry.attrs["gaiaxpy_metadata"]` in memory or the `_metadata.csv` companion file after saving to CSV. Only CSV produces companion files. For other formats, the metadata is contained in the output file itself. The metadata records the Gaia release and configuration used. For DR3 standardised photometry, it identifies the filter configuration and its flux-bias offsets. See [Standardised photometry](#gaiaxpy-standardised-photometry) for a worked example.